# 15_01 — Simulación y preprocesamiento · Escenario 5 (Algoritmo 5 del anexo)

Paso **1 de 3** del ciclo `Python → MATLAB → Python`:

| Paso | Archivo | Qué hace |
|---|---|---|
| 1 | **este notebook** | genera los datos, ajusta base + FPCA + estandarizador, escribe los datasets AR y todos los artefactos |
| 2 | `psbp_fd_iteracion.m` | muestreo MCMC en MATLAB, sólo con el bloque de entrenamiento |
| 3 | `15_03_convergencia` / `15_04_evaluacion` | diagnóstico de cadenas y evaluación fuera de muestra |

Las celdas marcadas **`[CONFIG]`** son las únicas que se tocan al cambiar de
experimento. Todo lo demás se deriva de ellas.

## Este escenario pertenece al Bloque 2, y eso cambia todo

Los Algoritmos 1 a 4 intervienen la **ley condicional** dejando intactos los
supuestos de la representación, y ahí la comparación relevante es PSBPM-FD
*vs.* referencias. El Algoritmo 5 hace lo contrario: deja la ley condicional
dentro de lo representable —es gaussiana, lineal, homogénea— y compromete la
**reducción de dimensión**.

El proceso se especifica directamente sobre los coeficientes de un sistema
ortonormal fijo, la base de Fourier con $J=10$:

$$X_t(\tau)=\mu(\tau)+\sum_{j=1}^{J}a_{tj}\,\phi_j(\tau),\qquad
a_{tj}=\varphi_j a_{t-1,j}+\eta_{tj},\quad
\eta_{tj}\sim N\bigl(0,\lambda_j(1-\varphi_j^2)\bigr).$$

Las $J$ recursiones son **escalares e independientes**. No hay operador
integral, ni factorización de Cholesky de una covarianza funcional, ni norma de
Hilbert-Schmidt: la curva sale de un único producto matricial
$X=\mathbf{1}\mu^{\top}+A\Phi^{\top}$. Lo único que este escenario comparte con
los anteriores es el **esquema de observación**.

El mecanismo es un **desalineamiento deliberado** entre dos órdenes:

- el espectro es geométrico, $\lambda_j=0.5^{\,j-1}$, de modo que el orden de
  **varianza** es $1>2>3>\dots$;
- la dinámica está concentrada en una sola componente, $\varphi_3=0.9$ y
  $\varphi_j=0$ para todo $j\neq 3$, de modo que el orden de
  **predictibilidad** tiene un único elemento: la tercera.

Que $\varphi_j=0$ **exacto** en las componentes dominantes no es cosmético: las
convierte en ruido blanco marginalmente independiente, de modo que ningún
método —bien o mal especificado— puede extraer predicción alguna de ellas. El
escenario queda así libre de la ambigüedad de atribuir el mal desempeño a una
dinámica débil en lugar de al truncamiento.

## Cómo se lee esta corrida, y una advertencia sobre $M$

**El resultado NO discrimina entre especificaciones dinámicas.** Si el
truncamiento descarta la dirección informativa, la degradación alcanza por igual
al PSBPM-FD, al FAR(1), al VAR sobre scores y a cualquier otro método sobre la
misma representación. Lo que este escenario acota es el **alcance de la
reducción de dimensión**, no la calidad de un modelo frente a otro. Hay que
decirlo así al reportar.

**Y aquí está el punto crítico de esta corrida.** El estudio fija `M = 4` como
invariante, para que $M$ no sea un segundo factor en la comparación entre los
seis algoritmos. Con $M=4$ la varianza acumulada es del 93.8 % y el truncamiento
**retiene** la componente informativa $j^{*}=3$. Es decir: **a este $M$ el
escenario es un caso nulo**, porque su tesis es que la dirección informativa es
una de las que el truncamiento descarta, y aquí no la descarta.

Eso no invalida la corrida —un caso nulo bien documentado es un resultado
legítimo, y sirve de control: confirma que sin truncamiento nocivo el método
recupera la dinámica— pero **no es lo que pide `docs/03 Modelo.tex §03_06`**.
Para que el escenario muerda hace falta $M\leq 2$, punto en el cual la varianza
acumulada sigue siendo un respetable 75 % y la componente predecible queda
fuera.

`M_FPCA` es por eso la única constante de este notebook marcada como
**`[BARRIDO]`**: está aislada en su propia celda para que bajarla a 2 y volver a
correr sea un cambio de un carácter. Al hacerlo hay que cambiar también el
`EXPERIMENT_ID`, porque la convención vigente no distingue corridas por $M$ y
los artefactos se sobrescribirían.

## 1. Imports y rutas

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Generadores y contrato de artefactos
from model_psbp_fd.pipelines import (
    ConfigEscenario5, generar_escenario_5, guardar_escenario,
    guardar_curvas, guardar_representacion, guardar_fpca,
    guardar_estandarizador, guardar_datasets_ar,
    guardar_hiperparametros, guardar_config_evaluacion,
    verificar_contrato,
)
# Preprocesamiento funcional
from model_psbp_fd.functions_models import (
    FunctionalRepresentation, FPCA_L2, base_en_grilla, DataStandardizer,
)
from model_psbp_fd.fit import tabla_baselines
from model_psbp_fd.utils import get_project_root
from model_psbp_fd.utils.quadrature import pesos_trapezoidales
from model_psbp_fd.graphics import (
    plot_empirical_sample, plot_mean_and_variance, plot_fts_empirical,
    plot_fts_functional, plot_diagnostico_estandarizacion, plot_fpca_scree,
    plot_seleccion_basis, plot_rezagos_heatmap,
)

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

### 1.1 `[CONFIG]` Identificación del experimento

In [ ]:
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASENAME     = "escenario"
ESCENARIO_ID = 5      # Algoritmo k del anexo
REPLICA_ID   = 1      # réplica Monte Carlo; eje del barrido en la Etapa D
SEED         = 41232  # semilla base; MATLAB la LEE de hyperparameters.json

EXPERIMENT_ID = f"{BASENAME}_{ESCENARIO_ID}_r{REPLICA_ID:02d}"

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"EXPERIMENT_ID : {EXPERIMENT_ID}")
print(f"Escenario {ESCENARIO_ID} · réplica {REPLICA_ID} · seed base {SEED}")
print("\nSi se repite esta corrida con otro M para hacer morder el escenario "
      "(ver §3.4),\nhay que cambiar EXPERIMENT_ID: la convención vigente NO "
      "distingue corridas por M\ny los artefactos se sobrescribirían. La "
      "convención para ese caso es materia de\nla Etapa D y todavía no está "
      "fijada.")

In [ ]:
# Las cinco rutas del contrato. No existe un config_paths en Python: cada
# notebook lo arma aquí y config_paths.m replica las mismas del lado MATLAB.
PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,
    "out_report":   PROJECT_ROOT / "reports" / "simulaciones" / EXPERIMENT_ID,
    "out_artefact": PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID,
}
for nombre, ruta in PATHS.items():
    ruta.mkdir(parents=True, exist_ok=True)
    print(f"  {nombre:12s} → {ruta}")

## 2. Simulación

### 2.1 `[CONFIG]` Parámetros del generador

**Atención con los defaults.** A diferencia de `ConfigEscenario1` a
`ConfigEscenario4`, las dataclasses de los Algoritmos 5 y 6 **redefinen** los
valores por defecto del esquema de observación para que
`ConfigEscenario5()` sin argumentos sea el escenario del Cuadro de escenarios
del Capítulo 3: `L=48`, `T=300`, `sigma_obs=0.1`, `R=50`. Ninguno de esos es el
valor del estudio, de modo que **los seis campos del esquema de observación hay
que pasarlos explícitos**. Omitir uno solo rompería la comparabilidad con las
corridas 11 a 14 sin dar ningún error.

Los parámetros propios del Algoritmo 5 provienen del Cuadro `tab:ane_alg5` y
coinciden con los defaults de la dataclass; se pasan explícitos de todos modos,
para que la corrida quede autodocumentada en el JSON y no dependa de cuál sea el
default vigente del código.

`J = 10` no es un parámetro del mecanismo sino de la **representación**: define
la dimensión exacta del espacio en que vive el proceso. Un FPCA con $M=J$
tendría error de representación nulo salvo ruido de medición; toda la
degradación de este escenario viene de que $M<J$ y de *cuáles* componentes
quedan dentro.

In [ ]:
# ── Parámetros fijos del estudio (comunes a los 6 escenarios) ────────────────
L_GRILLA   = 75     # puntos de la grilla regular tau_1=0 … tau_L=1
T_CURVAS   = 400    # curvas retenidas tras el calentamiento
PROP_TRAIN = 0.70   # proporción del bloque de entrenamiento
SIGMA_OBS  = 0.25   # desviación del ruido de medición

def media_senoidal(tau):
    """mu(tau) = sin(2 pi tau), Cuadro tab:ane_esquema. Con nombre para que
    quede legible en el JSON."""
    return np.sin(2.0 * np.pi * tau)

SIM_CFG = ConfigEscenario5(
    # Esquema de observación. TODOS explícitos: los defaults de esta dataclass
    # son los del Cuadro del Capítulo 3 (L=48, T=300, sigma_obs=0.1, R=50) y
    # NO son los del estudio.
    L         = L_GRILLA,
    T         = T_CURVAS,
    burn_in   = 200,
    sigma_obs = SIGMA_OBS,
    R         = 1,            # una réplica por corrida; el barrido usa REPLICA_ID
    seed      = SEED,
    media_fn  = media_senoidal,
    # Sistema ortonormal y espectro (Cuadro tab:ane_alg5)
    J              = 10,
    razon_espectro = 0.5,     # lambda_j = 0.5^(j-1)
    lambdas        = None,    # None ⇒ espectro geométrico
    # Dinámica: UNA sola componente predecible
    phis              = None,  # None ⇒ phi_j = 0 salvo en indice_predecible
    indice_predecible = 3,     # BASE-1 sobre la base de Fourier
    phi_predecible    = 0.9,
)

for k, v in SIM_CFG.to_dict().items():
    print(f"  {k:<24}: {v}")

# Verificación explícita de que los defaults NO se colaron.
assert (SIM_CFG.L, SIM_CFG.T, SIM_CFG.sigma_obs, SIM_CFG.R) == (75, 400, 0.25, 1), \
    ("Algún parámetro del esquema de observación quedó con el default de "
     "ConfigEscenario5 (L=48, T=300, sigma_obs=0.1, R=50) en vez del valor del "
     "estudio.")
print("\n  Esquema de observación sobrescrito correctamente (no quedó ningún "
      "default).")

### 2.2 Generación

In [ ]:
salida = generar_escenario_5(SIM_CFG)

REPLICA_IDX = REPLICA_ID - 1
X_raw   = salida.observaciones[REPLICA_IDX]   # (T, G) OBSERVADA — alimenta la estimación
X_true  = salida.curvas[REPLICA_IDX]          # (T, G) VERDADERA — objetivo de evaluación
A_coef  = salida.internos["coeficientes"][REPLICA_IDX]   # (T, J) coeficientes verdaderos
PHI_GEN = salida.internos["base"]                        # (G, J) base de Fourier
LAMBDAS = salida.internos["espectro"]                    # (J,)
PHIS    = salida.internos["coeficientes_ar"]             # (J,)
grilla  = salida.grilla
T, G    = X_raw.shape
J_GEN   = int(SIM_CFG.J)

print(f"Observadas {X_raw.shape} · verdaderas {X_true.shape} · grilla {grilla.shape}")
print(f"coeficientes {A_coef.shape} · base {PHI_GEN.shape}")
print(f"Ruido de medición efectivo: sd(X_raw - X_true) = {(X_raw - X_true).std():.4f}"
      f"   (nominal {SIGMA_OBS})")
print("\nControl de calidad del generador:")
for k, v in salida.diagnostico.items():
    if isinstance(v, list) and len(v) > 6:
        v = [round(float(x), 5) for x in v]
    print(f"  {k:36s} = {v}")

#### Lectura del control de calidad

Las tres verificaciones que este escenario necesita para ser el que dice ser:

- **`base_ortonormal`** y **`reproyeccion_error_max`**. Si $\Phi^{\top}W\Phi$ se
  aleja de la identidad, los coeficientes generados dejan de ser las coordenadas
  de la curva en la métrica de $L^2$ y todo el resto del diagnóstico pierde
  sentido. La condición que lo garantiza es $2k_{\max}<L-1$, verificada por
  `validar()`; con $L=75$ y $J=10$ se cumple con holgura.
- **`desalineacion_confirmada`**: que el índice de varianza máxima y el de
  predictibilidad máxima **no coincidan**. Si coincidieran, el escenario
  colapsaría al Algoritmo 1 y no sometería a prueba nada.
- **`componente_predecible_correcta`**: que la componente con mayor $|acf(1)|$
  empírica sea efectivamente $j^{*}$.

**`espectro_error_relativo_max` es informativo, no criterio.** Es la
discrepancia relativa entre la varianza empírica de cada componente y su
$\lambda_j$ objetivo; con $R=1$ y $T=400$ el error de muestreo de una varianza
es del orden de $\sqrt{2/T}\approx 7\,\%$, y en las componentes con $\varphi=0.9$
el tamaño muestral efectivo es mucho menor, de modo que un error relativo del
15–20 % en alguna componente es esperable y no indica un fallo de calibración.
Lo que sí sería un fallo es un error **sistemáticamente creciente en $j$**, señal
de que la corrección $(1-\varphi_j^2)$ no se aplicó.

In [ ]:
_d = salida.diagnostico

# ── ESTRUCTURALES: si fallan, el generador no es el del anexo ────────────────
_estructurales = [
    ("trayectorias finitas",              _d["todo_finito"],
     f"{_d['n_replicas']}x{_d['n_curvas']}x{_d['n_puntos_grilla']}"),
    ("base ortonormal en la métrica L2",  _d["base_ortonormal"],
     f"error max {_d['ortonormalidad_error_max']:.3e}"),
    ("reproyeccion exacta de A",          _d["reproyeccion_error_max"] < 1e-8,
     f"error max {_d['reproyeccion_error_max']:.3e}"),
    ("desalineamiento confirmado",        _d["desalineacion_confirmada"],
     f"varianza max en j={_d['indice_varianza_maxima']} · "
     f"predictibilidad max en j={_d['indice_predecibilidad_maxima']}"),
    ("componente predecible correcta",    _d["componente_predecible_correcta"],
     f"objetivo j*={_d['indice_predecible_objetivo']}"),
]
for nombre, ok, detalle in _estructurales:
    print(f"  {'OK ' if ok else 'FALLA'}  {nombre:34s} {detalle}")

assert all(ok for _, ok, _ in _estructurales), \
    "El generador no cumple las condiciones estructurales del Algoritmo 5."

# ── INFORMATIVOS: con R = 1 son cifras ruidosas; se reportan, no se exigen ───
print("\nEspectro y dinámica por componente de Fourier (informativo, R = 1):")
print(f"  {'j':>3} {'lambda_j':>10} {'var emp':>10} {'err rel':>9} "
      f"{'phi_j':>7} {'acf1 emp':>9}")
_acf1 = _d["acf1_por_componente"]
_vemp = _d["espectro_empirico"]
for j in range(J_GEN):
    marca = "  <- j* (única con dinámica)" if PHIS[j] != 0 else ""
    print(f"  {j+1:>3} {LAMBDAS[j]:>10.5f} {_vemp[j]:>10.5f} "
          f"{abs(_vemp[j]-LAMBDAS[j])/LAMBDAS[j]:>8.1%} "
          f"{PHIS[j]:>7.2f} {_acf1[j]:>9.4f}{marca}")

print(f"\nerror relativo máximo del espectro: {_d['espectro_error_relativo_max']:.1%}")
print(f"  Con R = 1 y T = {T}, un error de muestreo de una varianza es del orden "
      f"de\n  sqrt(2/T) = {np.sqrt(2/T):.1%}; en la componente con phi = "
      f"{SIM_CFG.phi_predecible} el tamaño muestral\n  efectivo es mucho menor. "
      f"Informativo, no criterio de aceptación.")

# Lo que SÍ sería un fallo: error creciente de forma sistemática en j.
_err_rel = np.abs(np.asarray(_vemp) - LAMBDAS) / LAMBDAS
_tend = float(np.corrcoef(np.arange(1, J_GEN + 1), _err_rel)[0, 1])
print(f"\ncorrelación (j, error relativo) = {_tend:+.3f}")
if _tend > 0.7:
    print("[AVISO] El error crece de forma sistemática con j: revisar que la "
          "corrección\n  (1 - phi_j^2) de la varianza de innovación se esté "
          "aplicando.")

### 2.3 El rasgo del escenario: varianza contra predictibilidad

Figura propia de los Algoritmos 5 y 6, sin equivalente en el Bloque 1. Hace
visible el **desalineamiento** entre los dos órdenes, que es el mecanismo
completo del escenario.

Nótese que la figura se dibuja sobre los **coeficientes verdaderos** $a_{tj}$ en
la base de Fourier del generador, no sobre los scores FPCA. Son objetos
distintos: los primeros son la verdad; los segundos, lo que el pipeline estima.
Que se correspondan es una pregunta empírica, y se responde en §3.3.

In [ ]:
w_quad  = pesos_trapezoidales(grilla)
acf1_A  = np.array([float(np.corrcoef(A_coef[1:, j], A_coef[:-1, j])[0, 1])
                    for j in range(J_GEN)])
var_A   = A_coef.var(axis=0)
var_cum_A = np.cumsum(var_A) / var_A.sum()
J_ESTRELLA = int(SIM_CFG.indice_predecible)      # base-1

fig = plt.figure(figsize=(13.5, 7.0))
gs  = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.26)
_jx = np.arange(1, J_GEN + 1)
_col = ["#c0392b" if PHIS[j] != 0 else "#2980b9" for j in range(J_GEN)]

# (a) espectro: el orden de varianza
ax = fig.add_subplot(gs[0, 0])
ax.bar(_jx, var_A, color=_col, alpha=0.85)
ax.plot(_jx, LAMBDAS, "k.--", ms=6, lw=1, label=r"$\lambda_j$ objetivo")
ax.set_yscale("log"); ax.set_xticks(_jx)
ax.set_xlabel("componente de Fourier $j$"); ax.set_ylabel("varianza")
ax.legend(fontsize=8)
ax.set_title("Orden de VARIANZA: decreciente por construcción", fontsize=10)

# (b) predictibilidad: el otro orden
ax = fig.add_subplot(gs[0, 1])
ax.bar(_jx, np.abs(acf1_A), color=_col, alpha=0.85)
ax.plot(_jx, np.abs(PHIS), "k.--", ms=6, lw=1, label=r"$|\varphi_j|$ objetivo")
ax.set_xticks(_jx); ax.set_ylim(0, 1)
ax.set_xlabel("componente de Fourier $j$"); ax.set_ylabel(r"$|acf(1)|$")
ax.legend(fontsize=8)
ax.set_title(f"Orden de PREDICTIBILIDAD: todo en $j^*={J_ESTRELLA}$", fontsize=10)

# (c) varianza acumulada y dónde caería el corte
ax = fig.add_subplot(gs[1, 0])
ax.plot(_jx, var_cum_A, "o-", color="#2c3e50", lw=1.4)
ax.axhline(0.95, color="k", ls=":", lw=1); ax.text(J_GEN, 0.95, " 95%", fontsize=8)
ax.axvline(J_ESTRELLA, color="#c0392b", ls="--", lw=1.4)
ax.text(J_ESTRELLA, 0.05, f" $j^*$={J_ESTRELLA}", color="#c0392b", fontsize=9)
ax.set_xticks(_jx); ax.set_ylim(0, 1.02)
ax.set_xlabel("componentes retenidas"); ax.set_ylabel("varianza acumulada")
ax.set_title("Un criterio de % de varianza corta ANTES de $j^*$ si el umbral "
             "es bajo", fontsize=10)

# (d) la componente informativa contra la dominante
ax = fig.add_subplot(gs[1, 1])
_n = min(200, T)
ax.plot(np.arange(1, _n + 1), A_coef[:_n, 0], lw=0.9, color="#2980b9",
        label=f"$a_{{t,1}}$ (var {var_A[0]:.3f}, $\\varphi$=0)")
ax.plot(np.arange(1, _n + 1), A_coef[:_n, J_ESTRELLA - 1], lw=1.2, color="#c0392b",
        label=f"$a_{{t,{J_ESTRELLA}}}$ (var {var_A[J_ESTRELLA-1]:.3f}, "
              f"$\\varphi$={PHIS[J_ESTRELLA-1]})")
ax.set_xlabel("$t$"); ax.legend(fontsize=8)
ax.set_title("La grande es ruido blanco; la predecible es pequeña", fontsize=10)

fig.suptitle("Escenario 5 — el orden de varianza y el de predictibilidad no "
             "coinciden", fontsize=12)
fig.savefig(PATHS["out_report"] / "10_desalineamiento.png", dpi=150,
            bbox_inches="tight")
plt.show()

print(f"varianza acumulada por número de componentes retenidas:")
for m in range(1, min(7, J_GEN) + 1):
    dentro = "SÍ" if m >= J_ESTRELLA else "NO"
    print(f"  M={m}: {var_cum_A[m-1]:.4%}   ¿contiene j*={J_ESTRELLA}? {dentro}")
print(f"\nCon M >= {J_ESTRELLA} el truncamiento RETIENE la dirección informativa "
      f"y el escenario\nno prueba su tesis. Con M < {J_ESTRELLA} la descarta y "
      f"el escenario muerde. Ver §3.4.")

#### Persistencia de la estructura verdadera

Se guarda como CSV —y no sólo dentro del `.npz`— porque es el insumo de la
sección propia de `15_04` y de la declaración de `VERDAD` en `15_03 §6.1`, y
conviene que sea legible y versionable sin abrir un binario.

**No es un estado latente por el cual estratificar**, a diferencia del régimen
de la corrida 13. Es una propiedad constante del proceso: qué componente lleva
la dinámica y cuánta varianza tiene cada una. Por eso `eval_config.json` declara
`estratificacion: null`.

In [ ]:
estructura_df = pd.DataFrame({
    "j_fourier":      np.arange(1, J_GEN + 1),   # base-1
    "lambda_objetivo": LAMBDAS,
    "var_empirica":   var_A,
    "var_ratio":      var_A / var_A.sum(),
    "var_acum":       var_cum_A,
    "phi_objetivo":   PHIS,
    "acf1_empirica":  acf1_A,
    "es_predecible":  PHIS != 0.0,
})
estructura_df.to_csv(PATHS["out_report"] / "10_estructura_generador.csv", index=False)
print(f"[out_report] 10_estructura_generador.csv  {estructura_df.shape}")
display(estructura_df.style.format({
    "lambda_objetivo": "{:.5f}", "var_empirica": "{:.5f}",
    "var_ratio": "{:.4%}", "var_acum": "{:.4%}",
    "phi_objetivo": "{:+.2f}", "acf1_empirica": "{:+.4f}"}))

In [ ]:
simulation_config = {
    "sim_params":    salida.config.to_dict(),
    "diagnostico":   salida.diagnostico,
    "replica_idx":   REPLICA_IDX,
    "experiment_id": EXPERIMENT_ID,
    "escenario_id":  int(ESCENARIO_ID),
    "replica_id":    int(REPLICA_ID),
    "seed":          SEED,
    "T": int(T), "G": int(G), "J": int(J_GEN),
    "indice_predecible": int(J_ESTRELLA),
}
with open(PATHS["raw"] / "simulation_config.json", "w", encoding="utf-8") as f:
    json.dump(simulation_config, f, indent=2, ensure_ascii=False)

# incluir_internos=True guarda `interno_coeficientes` (los a_tj verdaderos),
# `interno_base` (la base de Fourier), `interno_espectro` e
# `interno_coeficientes_ar` (los phi_j). Son la verdad contra la cual 15_03 §6.1
# declara VERDAD y 15_04 mide la degradación por truncamiento.
_npz = guardar_escenario(salida, str(PATHS["raw"] / f"escenario_{ESCENARIO_ID}"),
                         incluir_curvas=True, incluir_internos=True)
print(f"[raw] simulation_config.json  ·  {_npz}")

### 2.4 Visualización de los datos

In [ ]:
highlight_idx = [0, 1, T // 2, T - 1]

plot_fts_empirical(
    X_raw, grilla, highlight_idx=highlight_idx, separator_every=5,
    title=f"Predictibilidad en componente subordinada — {T} curvas observadas",
    save_path=str(PATHS["out_report"] / "01_fts_empirica_raw.png"))
plt.show()

plot_empirical_sample(
    X_raw, grilla, sample_idx=[0, 40, 80, 200, T - 1],
    title="Muestra de 5 curvas observadas",
    save_path=str(PATHS["out_report"] / "02_muestra_empirica_raw.png"))
plt.show()

plot_mean_and_variance(
    X_raw, grilla, show_std1=True, show_std2=True,
    title="Media y varianza funcional — Algoritmo 5",
    save_path=str(PATHS["out_report"] / "03_media_varianza_raw.png"))
plt.show()

In [ ]:
# Curva observada vs verdadera: dimensiona el ruido que el modelo NO debe predecir
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, i in zip(axes, [0, T // 2, T - 1]):
    ax.plot(grilla, X_raw[i], ".", color="0.65", ms=3, label="observada (con ruido)")
    ax.plot(grilla, X_true[i], color="#c0392b", lw=1.6, label="verdadera $X_t(\\tau)$")
    ax.set_title(rf"$t={i+1}$", fontsize=10)
    ax.set_xlabel(r"$\tau$")
axes[0].set_ylabel(r"$X_t(\tau)$"); axes[0].legend(fontsize=8)
fig.suptitle("Curva verdadera vs datos observados — el error se mide contra la primera",
             fontsize=12)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "04_curva_vs_datos.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.5 Persistencia de curvas

Se guardan las dos matrices con nombres distintos (`X_curves.npy` y
`X_curves_true.npy`) para que no puedan confundirse aguas abajo.

In [ ]:
_p = guardar_curvas(PATHS, X_raw, grilla, X_true=X_true)
for clave, ruta in _p.items():
    print(f"[functional] {clave:12s} → {ruta.name}")

### 2.6 Partición temporal

Todo objeto **estimado a partir de los datos** —selección GCV de la base, FPCA,
estandarizador— se ajusta sólo con $\{1,\dots,T_0\}$ y se aplica al bloque de
prueba mediante `transform`.

El proceso es estacionario y homogéneo en el tiempo, de modo que —al contrario
que en el Algoritmo 6— la base ajustada en entrenamiento **sigue siendo válida**
sobre el bloque de prueba. Lo que este escenario compromete no es la vigencia de
la base sino **cuántas direcciones se retienen**.

In [ ]:
T0 = int(np.floor(PROP_TRAIN * T))
assert 10 < T0 < T, f"T0={T0} fuera de rango para T={T}."

idx_train, idx_test = np.arange(0, T0), np.arange(T0, T)
X, X_train, X_test = X_raw, X_raw[idx_train], X_raw[idx_test]

print(f"entrenamiento : t en [1, {T0}]      → {X_train.shape}")
print(f"prueba        : t en [{T0+1}, {T}]  → {X_test.shape}")
print(f"proporción    : {T0/T:.1%} / {1 - T0/T:.1%}")

# El proceso es estacionario: el espectro empírico debe coincidir entre bloques.
_v_tr = A_coef[idx_train].var(axis=0)
_v_te = A_coef[idx_test].var(axis=0)
print(f"\nvarianza de los coeficientes verdaderos, por bloque:")
print(f"  {'j':>3} {'train':>9} {'test':>9} {'razón':>7}")
for j in range(min(5, J_GEN)):
    print(f"  {j+1:>3} {_v_tr[j]:>9.5f} {_v_te[j]:>9.5f} {_v_te[j]/_v_tr[j]:>7.2f}")
print("   Razones cercanas a 1 confirman la estacionariedad: la base ajustada "
      "en train\n   sigue siendo la correcta en test. En el Algoritmo 6 esto "
      "deja de cumplirse.")

## 3. Representación funcional B-spline

### 3.1 Barrido GCV sobre `(n_basis, order)`

El GCV **sugiere**; la elección es del analista y se declara en 3.2.

Nota propia del Bloque 2: el proceso vive exactamente en el span de $J=10$
funciones de Fourier, de modo que la base B-spline del estudio introduce un
error de representación **adicional** al del truncamiento FPCA. No es un
problema —es la misma base que usan las corridas 11 a 14, y ése es el
invariante— pero conviene tenerlo en cuenta al leer el `MISE_truncamiento` de
`15_04`: parte de él es B-spline y no FPCA.

In [ ]:
N_BASIS_RANGE = range(2, min(30, T0 // 2))   # acotado por T0
ORDER_RANGE   = range(2, 5)

registros = []
for orden in ORDER_RANGE:
    for nb in N_BASIS_RANGE:
        if nb < orden:
            continue
        try:
            fr_tmp = FunctionalRepresentation(method="bspline", n_basis=nb, order=orden)
            TH_tmp = fr_tmp.fit_transform(X_train, grilla)     # sólo train
            X_rec  = fr_tmp.reconstruct(TH_tmp)

            L_i    = X_train.shape[1]
            sse_c  = np.sum((X_train - X_rec) ** 2, axis=1)
            ss_tot = np.sum((X_train - X_train.mean(axis=0, keepdims=True)) ** 2)
            gcv_c  = (L_i * sse_c / (L_i - nb) ** 2 if L_i > nb
                      else np.full_like(sse_c, np.nan))
            registros.append({
                "n_basis": nb, "order": orden,
                "var_retained": 1.0 - sse_c.sum() / ss_tot,
                "rmse_mean": np.sqrt(sse_c / L_i).mean(),
                "rmse_max":  np.sqrt(sse_c / L_i).max(),
                "gcv_mean":  float(np.mean(gcv_c)),
            })
        except Exception as e:
            print(f"  [SKIP] n_basis={nb}, order={orden}: {e}")

sel_df   = pd.DataFrame(registros)
best_row = sel_df.dropna(subset=["gcv_mean"]).nsmallest(1, "gcv_mean").iloc[0]
nb_best, ord_best = int(best_row["n_basis"]), int(best_row["order"])

display(sel_df.style
    .format({"var_retained": "{:.4%}", "rmse_mean": "{:.6f}",
             "rmse_max": "{:.6f}", "gcv_mean": "{:.6f}"})
    .background_gradient(subset=["gcv_mean"], cmap="YlOrRd_r")
    .background_gradient(subset=["var_retained"], cmap="YlGn"))

print(f"\nGCV mínimo → n_basis={nb_best}, order={ord_best}  "
      f"(var retenida {best_row['var_retained']:.4%})")

plot_seleccion_basis(sel_df, nb_best, ord_best,
                     save_path=str(PATHS["out_report"] / "05_seleccion_basis.png"))
plt.show()

### 3.2 `[CONFIG]` Base elegida y ajuste

`center=False` es imprescindible: con `center=True` la reconstrucción es un
mapa **afín**, y la función media contaminaría la base recuperada por
`base_en_grilla`, la matriz de Gram y las autofunciones.

Se conserva la misma base que las corridas 11 a 14 (`n_basis=8`, `order=3`),
que es un invariante del estudio. Aquí conviene además mirar la cifra de error
de representación que se imprime abajo: si la base B-spline no resolviera los
armónicos altos de Fourier, la degradación que `15_04` atribuya al truncamiento
FPCA estaría contaminada.

In [ ]:
NB_ELEGIDO  = 8     # ← decisión del analista
ORD_ELEGIDO = 3

print(f"Base elegida : n_basis={NB_ELEGIDO}, order={ORD_ELEGIDO}")
print(f"Sugerido GCV : n_basis={nb_best}, order={ord_best}"
      + ("   (coinciden)" if (NB_ELEGIDO, ORD_ELEGIDO) == (nb_best, ord_best)
         else "   ← DIFIERE de la sugerencia; justificar en la tesis"))

fr = FunctionalRepresentation(method="bspline", n_basis=NB_ELEGIDO,
                              order=ORD_ELEGIDO, center=False)
fr.fit(X_train, grilla)                       # sólo train
THETA       = fr.transform(X, grilla)         # (T, K) serie completa
THETA_train = THETA[idx_train]
print(f"THETA {THETA.shape}  (train={T0}, test={T - T0})")

# Cuánto error de representación aporta la propia base B-spline, ANTES de
# cualquier truncamiento FPCA. Se mide contra la curva VERDADERA.
_Xrec = fr.reconstruct(THETA)
_mise_bspline = float((((X_true - _Xrec) ** 2) @ w_quad).mean())
print(f"\nMISE de la base B-spline contra la curva verdadera: {_mise_bspline:.6f}")
print("   Es un piso ADICIONAL al del truncamiento FPCA, propio del Bloque 2: "
      "el proceso\n   vive en el span de Fourier y la base del estudio es "
      "B-spline. 15_04 lo separa.")

plot_fts_functional(
    X, grilla, fr=fr, highlight_idx=highlight_idx, separator_every=5,
    title=f"Algoritmo 5 — repr. B-spline (n_basis={NB_ELEGIDO}, order={ORD_ELEGIDO})",
    save_path=str(PATHS["out_report"] / "06_fts_funcional_bspline.png"))
plt.show()

guardar_representacion(PATHS, fr, THETA,
    extra={"T0": int(T0), "prop_train": float(PROP_TRAIN), "ajustado_en": "train",
           "center": bool(fr.center), "n_basis": int(NB_ELEGIDO),
           "order": int(ORD_ELEGIDO), "n_basis_gcv": nb_best, "order_gcv": ord_best,
           "mise_bspline_vs_verdadera": _mise_bspline})
print("[functional] functional_representation.pkl + theta.csv + fr_config.json")

### 3.3 FPCA generalizado, y **el diagnóstico propio del escenario**

La base B-spline no es ortonormal, de modo que la Gram $W\neq I$ y la
descomposición correcta resuelve el problema propio generalizado
$(W^{1/2}S_\theta W^{1/2})z=\lambda z$. Se ajusta **sólo con entrenamiento**.

**Ésta es la sección que hace visible el escenario.** El FPCA ordena las
direcciones por varianza; el generador puso la dinámica en una dirección
subordinada. La tabla de abajo pone las dos cosas en columnas contiguas:

- `var_ratio`: cuánta varianza aporta cada componente FPCA. Es el criterio por
  el que se trunca.
- `ar1_propio`: la autocorrelación a rezago uno del score. Es lo que hace
  predecible a la componente, y es lo que el criterio de truncamiento **no
  mira**.

Se añade además la **alineación** entre cada componente FPCA estimada y las
componentes de Fourier del generador, medida por correlación entre el score y el
coeficiente verdadero $a_{tj}$. No se supone: se calcula, y de ella se deriva
`VERDAD` en `15_03 §6.1`. Si el FPCA recupera el orden de Fourier —lo esperable,
porque el espectro es estrictamente decreciente— la componente FPCA $k$ debería
alinearse con la $k$-ésima de Fourier.

In [ ]:
Phi  = base_en_grilla(fr, THETA.shape[1])        # (G, K)
fpca = FPCA_L2().fit(THETA_train, Phi, grilla)

_ver = fpca.verificar(THETA_train, fr=fr)
print("Verificación FPCA_L2 (entrenamiento):")
for k, v in _ver.items():
    print(f"  {k:34s} = {v:.3e}" if isinstance(v, float) else f"  {k:34s} = {v}")

cond = _ver["cond_W"]
nota = ("← MUY ALTO: reduzca n_basis" if cond > 1e10 else
        "← alto: vigile las componentes menores" if cond > 1e6 else "(sano)")
print(f"\ncond(W) = {cond:.3e}  {nota}")

assert _ver["todo_ok"], ("Las identidades del FPCA generalizado no se cumplen. "
                         "Si falla err_linealidad_reconstruct_rel, revise center=False.")

In [ ]:
# ── Varianza contra predictibilidad, componente FPCA por componente FPCA ─────
K_FPCA = fpca.evals.size
S_a = (THETA_train - fpca.mu_theta) @ (fpca.W @ fpca.B_full)     # (T0, K)
ar1 = (S_a[1:] * S_a[:-1]).sum(0) / np.clip((S_a[:-1] ** 2).sum(0), 1e-12, None)

# Alineación con la base del generador: NO se supone, se calcula.
A_tr = A_coef[idx_train]                                          # (T0, J)
corr_AF = np.zeros((K_FPCA, J_GEN))
for k in range(K_FPCA):
    for j in range(J_GEN):
        _s, _a = S_a[:, k], A_tr[:, j]
        if _s.std() < 1e-12 or _a.std() < 1e-12:
            corr_AF[k, j] = 0.0
        else:
            corr_AF[k, j] = abs(float(np.corrcoef(_s, _a)[0, 1]))

fourier_de_fpc = np.argmax(corr_AF, axis=1) + 1        # base-1
corr_max       = corr_AF.max(axis=1)
phi_de_fpc     = np.array([PHIS[j - 1] for j in fourier_de_fpc])

VAR_TARGET  = 0.95
M_SUGERIDO  = fpca.seleccionar_M(VAR_TARGET)

tabla_fpca = pd.DataFrame({
    "componente": np.arange(1, K_FPCA + 1),
    "autovalor": fpca.evals, "var_ratio": fpca.var_ratio, "var_acum": fpca.var_cum,
    "ar1_propio": ar1,
    "fourier_alineada": fourier_de_fpc, "corr_alineacion": corr_max,
    "phi_generador": phi_de_fpc,
})
display(tabla_fpca.head(min(15, K_FPCA)).style.format(
    {"autovalor": "{:.4e}", "var_ratio": "{:.4%}", "var_acum": "{:.4%}",
     "ar1_propio": "{:+.3f}", "corr_alineacion": "{:.4f}",
     "phi_generador": "{:+.2f}"})
    .background_gradient(subset=["var_ratio"], cmap="YlGn")
    .background_gradient(subset=["ar1_propio"], cmap="coolwarm", vmin=-1, vmax=1)
    .background_gradient(subset=["corr_alineacion"], cmap="Blues", vmin=0, vmax=1)
    .set_caption("Varianza vs predictibilidad por componente FPCA, y su "
                 "alineación con la base del generador"))

print(f"\nK disponibles: {K_FPCA}   ·   sugerencia (var >= {VAR_TARGET:.0%}): "
      f"M = {M_SUGERIDO}")
_k_var  = int(np.argmax(fpca.var_ratio))
_k_pred = int(np.nanargmax(np.abs(ar1)))
print(f"componente FPCA de MAYOR VARIANZA        : FPC {_k_var + 1}")
print(f"componente FPCA de MAYOR PREDICTIBILIDAD : FPC {_k_pred + 1} "
      f"(ar1 = {ar1[_k_pred]:+.3f})")
print(f"   Son distintas: {_k_var != _k_pred}   ← el desalineamiento sobrevive "
      f"al pipeline")

# ¿En qué componente FPCA cayó j*?
_k_estrella = int(np.argmax(corr_AF[:, J_ESTRELLA - 1]))
print(f"\nla componente de Fourier j*={J_ESTRELLA} (la única con dinámica) se "
      f"alinea con FPC {_k_estrella + 1}")
print(f"   correlación {corr_AF[_k_estrella, J_ESTRELLA - 1]:.4f}   ·   "
      f"var_ratio de esa FPC {fpca.var_ratio[_k_estrella]:.4%}")

assert _k_var != _k_pred, (
    "En la representación estimada, la componente de mayor varianza es también "
    "la de mayor predictibilidad: el desalineamiento no sobrevivió al pipeline "
    "y el escenario no prueba nada. Revisar el generador.")
print("\n  El desalineamiento entre varianza y predictibilidad se conserva en "
      "la representación estimada.")

plot_fpca_scree(fpca.evals, fpca.var_cum, M_SUGERIDO, var_target=VAR_TARGET,
                save_path=str(PATHS["out_report"] / "07_fpca_scree.png"))
plt.show()

In [ ]:
# Figura propia: var_ratio y ar1 lado a lado, con j* señalado. Es la imagen que
# resume el escenario y la que conviene llevar a la tesis.
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
_kx = np.arange(1, min(10, K_FPCA) + 1)
_n  = _kx.size
_col = ["#c0392b" if k == _k_estrella else "#2980b9" for k in range(_n)]

axes[0].bar(_kx, fpca.var_ratio[:_n], color=_col, alpha=0.85)
axes[0].set_yscale("log"); axes[0].set_xticks(_kx)
axes[0].set_xlabel("componente FPCA"); axes[0].set_ylabel("var_ratio")
axes[0].set_title("Criterio de truncamiento: varianza", fontsize=10)

axes[1].bar(_kx, np.abs(ar1[:_n]), color=_col, alpha=0.85)
axes[1].set_xticks(_kx); axes[1].set_ylim(0, 1)
axes[1].set_xlabel("componente FPCA"); axes[1].set_ylabel(r"$|ar1|$ del score")
axes[1].set_title("Lo que el criterio NO mira: predictibilidad", fontsize=10)

for ax in axes:
    ax.axvline(_k_estrella + 1, color="#c0392b", ls="--", lw=1.2, alpha=0.7)
fig.suptitle(f"Escenario 5 — la única componente con dinámica es la FPC "
             f"{_k_estrella + 1} (en rojo)", fontsize=12)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "11_varianza_vs_predictibilidad.png",
            dpi=150, bbox_inches="tight")
plt.show()

### 3.4 `[BARRIDO]` Componentes FPCA retenidas — la decisión crítica

`M_FPCA` es un **invariante del estudio** (`M = 4`, igual que las corridas 11 a
14) y a la vez el parámetro que decide si este escenario prueba algo. Las dos
cosas son incompatibles, y la celda de abajo lo declara en vez de esconderlo.

Con $M=4$:

- la varianza acumulada es alta y el ajuste es bueno;
- **la componente informativa queda dentro**, de modo que la degradación que el
  Algoritmo 5 existe para producir **no ocurre**;
- la corrida es un **caso nulo**, y como tal se reporta: sirve de control —el
  método recupera la dinámica cuando el truncamiento no la descarta— pero no
  responde la pregunta de `§03_06`.

Para que el escenario muerda hay que bajar `M_FPCA` por debajo del índice de la
componente que lleva la dinámica, que §3.3 acaba de identificar. La celda
imprime ese umbral: cambiar el número, cambiar el `EXPERIMENT_ID` y volver a
correr desde aquí es todo lo que hace falta.

In [ ]:
M_FPCA = 4   # ← [BARRIDO] invariante del estudio. Ver el aviso que imprime
             #   esta celda: para que el Algoritmo 5 muerda hace falta bajarlo.

assert 1 <= M_FPCA <= fpca.evals.size, f"M_FPCA fuera de [1, {fpca.evals.size}]."
fpca.set_M(int(M_FPCA))
M_fpca = fpca.M

Psi_grid, mu_grid = fpca.Psi_grid, fpca.mu_grid
SCORES       = fpca.transform(THETA)       # (T, M) — base ajustada en train
SCORES_train = SCORES[idx_train]
SCORES_test  = SCORES[idx_test]

print(f"M = {M_fpca}   var. explicada = {fpca.var_cum[M_fpca-1]:.4%}")
print(f"[train] max|media xi| = {np.abs(SCORES_train.mean(0)).max():.2e}   (aprox 0)")
print(f"[test]  max|media xi| = {np.abs(SCORES_test.mean(0)).max():.3f}")

# ── La pregunta que decide la corrida ────────────────────────────────────────
M_UMBRAL = _k_estrella + 1          # base-1: primera M que RETIENE j*
RETIENE  = M_fpca >= M_UMBRAL

print("\n" + "=" * 70)
print(f"¿El truncamiento a M={M_fpca} retiene la dirección informativa?  "
      f"{'SÍ' if RETIENE else 'NO'}")
print(f"  la dinámica del generador vive en FPC {M_UMBRAL} "
      f"(alineada con Fourier j*={J_ESTRELLA})")
print(f"  |ar1| de esa componente : {abs(ar1[_k_estrella]):.4f}")
print(f"  var_ratio de esa componente: {fpca.var_ratio[_k_estrella]:.4%}")
if RETIENE:
    print("\n  CASO NULO. A este M el escenario NO prueba su tesis: la "
          "dirección\n  informativa está dentro del truncamiento, de modo que "
          "no hay degradación\n  por reducción de dimensión que medir. La "
          "corrida es válida y reportable\n  como CONTROL —confirma que el "
          "método recupera la dinámica cuando el\n  truncamiento no la "
          f"descarta— pero para responder §03_06 hace falta\n  M <= "
          f"{M_UMBRAL - 1}.")
else:
    print("\n  EL ESCENARIO MUERDE. La dirección informativa quedó FUERA del "
          "truncamiento:\n  los scores retenidos son ruido blanco y ningún "
          "método sobre esta\n  representación puede predecir. Eso es "
          "exactamente lo que el Algoritmo 5\n  existe para mostrar.")
print("=" * 70)

# Cuánta predictibilidad sobrevive al truncamiento: la cifra a reportar.
_ar1_ret = np.abs(ar1[:M_fpca])
_ar1_des = np.abs(ar1[M_fpca:K_FPCA])
print(f"\n|ar1| máximo entre las componentes RETENIDAS  : {_ar1_ret.max():.4f}")
print(f"|ar1| máximo entre las componentes DESCARTADAS: "
      f"{(_ar1_des.max() if _ar1_des.size else float('nan')):.4f}")

## 4. Datasets AR($p$) sobre los scores

### 4.1 Estandarización

El estandarizador se ajusta **sólo con train** y registra `n_ajuste` para que
esa disciplina sea auditable desde el artefacto y no una promesa del notebook.

In [ ]:
scores_standardizer = DataStandardizer(method="zscore_column", ddof=0)
scores_standardizer.fit(SCORES_train, etiqueta=f"train[1:{T0}]")

_chk = scores_standardizer.verificar_ajuste(T0)
print(f"[holdout] ajustado con {_chk['n_ajuste']} filas = T0 "
      f"({_chk['etiqueta_ajuste']}) → ok={_chk['ajuste_ok']}")

SCORES_STD       = scores_standardizer.transform(SCORES)
SCORES_STD_train = SCORES_STD[idx_train]
SCORES_STD_test  = SCORES_STD[idx_test]

print(f"\nSCORES_STD {SCORES_STD.shape}")
print(f"  [train] max|media| = {np.abs(SCORES_STD_train.mean(0)).max():.2e}  (aprox 0)")
print(f"  [train] max|std-1| = {np.abs(SCORES_STD_train.std(0) - 1).max():.2e}  (aprox 0)")
print(f"  [test]  media = {np.array2string(SCORES_STD_test.mean(0), precision=3)}")
print(f"  [test]  std   = {np.array2string(SCORES_STD_test.std(0),  precision=3)}")

guardar_estandarizador(PATHS, scores_standardizer)
_res = guardar_fpca(PATHS, fpca, SCORES, SCORES_STD=SCORES_STD,
                    meta_extra={"T0": int(T0)})
print(f"\n[functional] artefactos FPCA · cond_W = {_res['meta']['cond_W']:.3e}")

plot_diagnostico_estandarizacion(
    SCORES_train, SCORES_STD_train, np.arange(1, M_fpca + 1),
    labels=("Scores xi (escala lambda)", "Scores xi estandarizados"),
    title="estadísticas por componente FPCA",
    save_path=str(PATHS["out_report"] / "08_diagnostico_estandarizacion.png"))
plt.show()

### 4.2 Diagnóstico de rezagos (sólo train)

Aquí los mapas de calor tienen una lectura muy concreta, distinta de la de todos
los escenarios anteriores: **deben estar casi vacíos salvo una única celda en la
diagonal**. Las recursiones del generador son escalares e independientes, de
modo que:

- no hay dependencia **cruzada** entre componentes: toda correlación fuera de la
  diagonal es ruido muestral;
- de la diagonal, sólo una entrada debe destacar, la de la componente alineada
  con $j^{*}$.

Ese patrón es la firma del Algoritmo 5, y es lo que el eje 3 (`15_03 §6`)
debería recuperar en las PIP.

In [ ]:
N_LAGS_MAX = 3
T_theta, K_total = SCORES_STD_train.shape

def _spearman(Y, Xm):
    """Spearman columna a columna vía rangos (pandas, sin scipy)."""
    Yc = pd.DataFrame(Y).rank().to_numpy(); Yc = Yc - Yc.mean(0)
    Xc = pd.DataFrame(Xm).rank().to_numpy(); Xc = Xc - Xc.mean(0)
    return (Yc.T @ Xc) / np.outer(np.sqrt((Yc**2).sum(0)), np.sqrt((Xc**2).sum(0)))

corr_p = np.zeros((K_total, K_total * N_LAGS_MAX))
corr_s = np.zeros_like(corr_p)
col_labels = []
y_block = SCORES_STD_train[N_LAGS_MAX:, :]

for lag in range(1, N_LAGS_MAX + 1):
    x_block = SCORES_STD_train[N_LAGS_MAX - lag : T_theta - lag, :]
    sp = _spearman(y_block, x_block)
    for j in range(K_total):
        c = (lag - 1) * K_total + j
        for k in range(K_total):
            corr_p[k, c] = np.corrcoef(y_block[:, k], x_block[:, j])[0, 1]
        corr_s[:, c] = sp[:, j]
        col_labels.append(rf"$\xi_{{t-{lag},{j+1}}}$")

row_labels = [rf"$\xi_{{t,{k+1}}}$" for k in range(K_total)]
band = 1.96 / np.sqrt(len(y_block))

for M_corr, nombre, arch in ((corr_p, "Pearson", "09a"), (corr_s, "Spearman", "09b")):
    plot_rezagos_heatmap(
        M_corr, col_labels, row_labels,
        title=f"{nombre} — respuesta($t$) vs rezagos 1..{N_LAGS_MAX}",
        n_lags_max=N_LAGS_MAX, K_total=K_total, band=band, vclip=0.6,
        save_path=str(PATHS["out_report"] / f"{arch}_rezagos_{nombre.lower()}.png"))
    plt.show()

# Panel propio del escenario: la diagonal a rezago 1, componente a componente.
print("correlación de cada score con su PROPIO rezago 1 (diagonal, lag 1):")
for k in range(K_total):
    _c = corr_p[k, k]
    marca = ("  <- la única con dinámica" if k == _k_estrella
             else ("  <- por encima de la banda" if abs(_c) > band else ""))
    print(f"  FPC {k+1}: {_c:+.4f}{marca}")
_fuera = np.abs(corr_p[:, :K_total]).copy()
np.fill_diagonal(_fuera, 0.0)
print(f"\nmáx |corr| CRUZADA a rezago 1: {_fuera.max():.4f}   "
      f"(banda de ruido {band:.4f})")
print("   El generador no tiene dependencia cruzada: cualquier valor por encima "
      "de la\n   banda es ruido muestral de una sola trayectoria.")

### 4.3 `[CONFIG]` Orden AR y construcción de los datasets

Los rezagos del primer origen de prueba vienen del final del bloque de
entrenamiento: son observaciones pasadas disponibles en cada origen, de modo
que su uso es el condicionamiento de la predicción a $h=1$, no fuga.

`N_LAGS = 1` se mantiene igual que en las corridas 11 a 14, y aquí queda además
justificado de forma exacta por el generador: cada coeficiente sigue un AR(1)
escalar, de modo que un rezago contiene **toda** la información disponible.

In [ ]:
N_LAGS        = 1
COMPONENT_IDX = list(range(SCORES_STD.shape[1]))   # base-0; los nombres usan idx+1

n_components = len(COMPONENT_IDX)
n_train_eff  = T0 - N_LAGS
n_test_eff   = T - T0

assert T0 > N_LAGS and len(set(COMPONENT_IDX)) == n_components

cov_names = [f"fpc_{COMPONENT_IDX[j] + 1}_lag{lag}"
             for lag in range(1, N_LAGS + 1)
             for j in range(n_components)]

print(f"componentes : {n_components} → índices {COMPONENT_IDX}")
print(f"N_LAGS      : {N_LAGS}   ·   p = {len(cov_names)} covariables")
print(f"n_train_eff : {n_train_eff}   n_test_eff : {n_test_eff}")
print(f"cov_names   : {cov_names}")

In [ ]:
SCORES_sel = SCORES_STD[:, COMPONENT_IDX]

def _dataset_bloque(k, t_ini, t_fin):
    """Respuesta en t en [t_ini, t_fin) y predictores en t-1 … t-N_LAGS."""
    t_idx  = np.arange(t_ini, t_fin)
    X_cols = np.hstack([SCORES_sel[t_idx - lag, :] for lag in range(1, N_LAGS + 1)])
    return pd.DataFrame(np.column_stack([SCORES_sel[t_idx, k], X_cols]),
                        columns=[f"fpc_{COMPONENT_IDX[k] + 1}"] + cov_names)

dfs_train = {k: _dataset_bloque(k, N_LAGS, T0) for k in range(n_components)}
dfs_test  = {k: _dataset_bloque(k, T0,     T)  for k in range(n_components)}

manifest = {
    "scores_scale":  "standardized_zscore_ddof0",
    "n_components":  n_components,
    "n_lags":        int(N_LAGS),
    "component_idx": [int(i) for i in COMPONENT_IDX],
    "cov_names":     cov_names,
    "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
    "n_train_eff": int(n_train_eff), "n_test_eff": int(n_test_eff),
    "ajuste_en": "train",
}
guardar_datasets_ar(PATHS, dfs_train, dfs_test, manifest)
print(f"[functional] {2*n_components} datasets + datasets_manifest.json")
for k in range(n_components):
    print(f"  fpc_{COMPONENT_IDX[k]+1}: train {dfs_train[k].shape} · test {dfs_test[k].shape}")

#### Persistencia de la alineación FPCA ↔ generador

Ésta es la tabla que `15_03 §6.1` lee para declarar `VERDAD` y que `15_04` usa
para medir la degradación. Se escribe **después** de fijar `M`, de modo que
incluye la columna `retenida`, que es lo que decide si el escenario muerde.

Se persiste en vez de recalcularse porque la alineación depende del FPCA
ajustado en entrenamiento, y `15_03` no vuelve a ajustarlo.

In [ ]:
alineacion_df = pd.DataFrame({
    "fpc":              np.arange(1, K_FPCA + 1),          # base-1
    "fourier_alineada": fourier_de_fpc,                    # base-1
    "corr_alineacion":  corr_max,
    "var_ratio":        fpca.var_ratio,
    "ar1_score":        ar1,
    "phi_generador":    phi_de_fpc,
    "es_predecible":    phi_de_fpc != 0.0,
    "retenida":         np.arange(1, K_FPCA + 1) <= M_fpca,
})
alineacion_df.to_csv(PATHS["out_report"] / "10_alineacion_fpca_generador.csv",
                     index=False)
print(f"[out_report] 10_alineacion_fpca_generador.csv  {alineacion_df.shape}")
display(alineacion_df.head(min(10, K_FPCA)).style.format({
    "corr_alineacion": "{:.4f}", "var_ratio": "{:.4%}",
    "ar1_score": "{:+.4f}", "phi_generador": "{:+.2f}"}))

# Verificación numérica contra la fuente, no contra la memoria: los phi que se
# atribuyen a cada FPC deben salir de internos["coeficientes_ar"].
assert np.allclose(phi_de_fpc, [PHIS[j - 1] for j in fourier_de_fpc]), \
    "phi_generador no coincide con internos['coeficientes_ar']."
_activas = alineacion_df.loc[alineacion_df.retenida & alineacion_df.es_predecible,
                             "fpc"].tolist()
print(f"\ncomponentes RETENIDAS con dinámica verdadera: {_activas}")
print(f"   (de ellas se deriva VERDAD en 15_03 §6.1)")

## 5. `[CONFIG]` Hiperparámetros y configuración MCMC

Esto es el **contrato con MATLAB**: `psbp_fd_iteracion.m` lee estos valores del
JSON y no los tiene escritos a mano, incluida `seed_base`.

Ojo con la doble acepción de `M`: aquí, dentro de `mcmc_config`, es el tamaño
de la grilla de localización $G^*$ del stick-breaking, **no** el número de
componentes FPCA. `N` es el truncamiento del número de átomos.

Priors y `mcmc_config` **idénticos a los de las corridas 11 a 14**. Aquí el
prior $E[\pi]=0.90$ sobre el propio rezago está **mal orientado en todas las
componentes salvo una**: el generador tiene $\varphi_j=0$ exacto en todas menos
$j^{*}$, de modo que en la mayoría de los modelos el prior empuja a incluir una
covariable que no hace nada. Que el modelo lo desmienta es parte de la prueba
del eje 3, igual que en la corrida 12, y por eso el prior no se cambia.

In [ ]:
MCMC_CONFIG = {"nsim": 2000, "burn": 500, "N": 35, "M": 35}
N_CHAINS    = 3

print(f"MCMC_CONFIG : {MCMC_CONFIG}")
print(f"N_CHAINS    : {N_CHAINS} cadenas por componente "
      f"→ {N_CHAINS * n_components} jobs en MATLAB")
print(f"Draws posteriores por score: "
      f"({MCMC_CONFIG['nsim']} - {MCMC_CONFIG['burn']}) x {N_CHAINS} = "
      f"{(MCMC_CONFIG['nsim'] - MCMC_CONFIG['burn']) * N_CHAINS}")
print(f"\nN = {MCMC_CONFIG['N']} átomos. El generador es gaussiano, lineal y "
      f"homogéneo: la ley\ncondicional de cada score es una ÚNICA normal. "
      f"Cabe esperar por tanto una\nocupación cercana a uno —la más baja de "
      f"todo el estudio— y eso sería el\nresultado CORRECTO, no un fallo "
      f"(ver 15_03 §4).")

In [ ]:
# Priors globales
HP_GLOBAL = {"atau": 2.0, "btau": 0.5, "ag": 2.0, "bg": 0.5,
             "mumu": 0.0, "taumu": 1.0, "pwj": 0.5}

# Priors por tipo de covariable: (apij, bpij, mupsij, taupsij)
HP_BY_TYPE = {
    "own_lag1":  (9.0, 1.0, 0.0, 1.0),   # E[pi] = 0.90 — el propio rezago 1
    "cross_lag": (1.0, 1.0, 0.0, 1.0),   # E[pi] = 0.50 — los cruzados
}

def _clasificar(nombre, k_modelo):
    return ("own_lag1" if nombre == f"fpc_{COMPONENT_IDX[k_modelo] + 1}_lag1"
            else "cross_lag")

HYPERPARAMS_LIST = []
for k in range(n_components):
    tipos = [_clasificar(nm, k) for nm in cov_names]
    vals  = np.array([HP_BY_TYPE[t] for t in tipos], dtype=float)   # (p, 4)
    HYPERPARAMS_LIST.append({**HP_GLOBAL,
        "apij": vals[:, 0], "bpij": vals[:, 1],
        "mupsij": vals[:, 2], "taupsij": vals[:, 3]})

for k in range(n_components):
    hp = HYPERPARAMS_LIST[k]
    print(f"\nComponente k={k}  (fpc_{COMPONENT_IDX[k]+1})")
    print(f"  {'variable':<22} {'tipo':<11} {'apij':>6} {'bpij':>6} {'E[pi]':>7}")
    for j, nm in enumerate(cov_names):
        a, b = hp["apij"][j], hp["bpij"][j]
        marca = "  <-" if _clasificar(nm, k) == "own_lag1" else ""
        print(f"  {nm:<22} {_clasificar(nm, k):<11} {a:>6.1f} {b:>6.1f} "
              f"{a/(a+b):>7.3f}{marca}")

In [ ]:
hp_artifact = {
    "global":       HP_GLOBAL,
    "by_type":      HP_BY_TYPE,
    "mcmc_config":  MCMC_CONFIG,
    "n_iter":       N_CHAINS,
    "seed_scheme":  "seed_base + chain*9973 + k*31",
    "escenario_id": int(ESCENARIO_ID),
    "replica_id":   int(REPLICA_ID),
    "seed_base":    int(SEED),     # MATLAB la lee de aquí (ya no está hardcodeada)
    "scores_scale": "standardized_zscore_ddof0",
    "partition": {
        "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
        "n_train_eff": int(n_train_eff), "n_test_eff": int(n_test_eff),
        "train_files": [f"dataset_fpc_{COMPONENT_IDX[k]+1}_train.csv"
                        for k in range(n_components)],
        "test_files":  [f"dataset_fpc_{COMPONENT_IDX[k]+1}_test.csv"
                        for k in range(n_components)],
    },
    "hyperparams_list": [
        {"component_k": k, "fpc_idx": int(COMPONENT_IDX[k] + 1),
         "hyperparams": {key: (v.tolist() if isinstance(v, np.ndarray) else v)
                         for key, v in HYPERPARAMS_LIST[k].items()}}
        for k in range(n_components)
    ],
}
guardar_hiperparametros(PATHS, hp_artifact)
print(f"OK  hyperparameters.json → {PATHS['out_artefact']}")

## 6. Configuración de evaluación, líneas base y verificación del contrato

`objetivo_evaluacion` y `modo_residuo` se declaran aquí porque cambian el
significado de toda la evaluación y no deben quedar como una decisión implícita
del notebook `_04`.

**`estratificacion` es `null`**, como en la corrida 14 y por una razón análoga:
el Algoritmo 5 no tiene estado latente. El proceso es estacionario y homogéneo;
lo que el escenario interviene es la representación, que es una propiedad del
diseño y no una variable que cambie de origen a origen.

El bloque **`representacion`** es el propio de este escenario y es lo que `15_04`
lee para su sección de diagnóstico: qué componente lleva la dinámica, si quedó
dentro del truncamiento y con cuánta varianza.

In [ ]:
eval_config = {
    "scheme":      "holdout_temporal",
    "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
    "horizons":    [1],
    "n_lags":      int(N_LAGS),
    "scores_scale": "standardized_zscore_ddof0",
    # Contra QUÉ se mide el error y con qué banda:
    #   curva_verdadera + modo_residuo "ninguno" ⇒ la banda cubre la curva
    #   PROYECTADA sobre las M autofunciones y se contrasta con X_t(tau).
    "objetivo_evaluacion": "curva_verdadera",
    "modo_residuo":        "ninguno",
    "nivel_credibilidad":  0.95,
    "ventana_movil": {"w": [10, 20, 40], "paso": 1, "solapadas": True},
    "metrics_scores": ["RMSE", "R2", "razon_dispersion"],
    "metrics_curvas": ["MISE", "RMSE_funcional"],
    "metrics_dist":   ["CRPS", "energy_score", "cobertura_95", "PIT"],
    # Eje 2: NO hay estado latente por el cual estratificar en este escenario.
    "estratificacion": None,
    "estratificacion_motivo": (
        "El Algoritmo 5 no tiene estado latente: el proceso es gaussiano, "
        "lineal, estacionario y homogéneo en el tiempo. Lo que el escenario "
        "interviene es la reducción de dimensión, que es una propiedad del "
        "diseño y no una variable que cambie de origen a origen. 15_04 omite "
        "la sección de calibración condicional, como las corridas 11 y 14."
    ),
    # Propio del Bloque 2: la representación es el objeto de estudio.
    "representacion": {
        "bloque_del_anexo":        2,
        "J_generador":             int(J_GEN),
        "base_generador":          "fourier ortonormal en L2",
        "indice_predecible_fourier": int(J_ESTRELLA),
        "phi_predecible":          float(SIM_CFG.phi_predecible),
        "phis_generador":          [float(p) for p in PHIS],
        "espectro_generador":      [float(l) for l in LAMBDAS],
        "M_retenidas":             int(M_fpca),
        "fpc_con_dinamica":        int(M_UMBRAL),
        "M_umbral_para_retener":   int(M_UMBRAL),
        "retiene_direccion_informativa": bool(RETIENE),
        "es_caso_nulo":            bool(RETIENE),
        "var_ratio_fpc_con_dinamica": float(fpca.var_ratio[_k_estrella]),
        "ar1_fpc_con_dinamica":    float(ar1[_k_estrella]),
        "mise_bspline_vs_verdadera": float(_mise_bspline),
        "fuente":                  "reports/.../10_alineacion_fpca_generador.csv",
        "respaldo":                f"raw/escenario_{ESCENARIO_ID}.npz::interno_coeficientes",
        "advertencia": (
            "El resultado de este escenario NO discrimina entre "
            "especificaciones dinámicas: la degradación por truncamiento "
            "alcanza por igual a todo método sobre la misma representación. "
            "Acota el alcance de la reducción de dimensión."
        ),
    },
}
guardar_config_evaluacion(PATHS, eval_config)
print("[out_artefact] eval_config.json")
for k, v in eval_config.items():
    print(f"  {k:26s}: {v}")

### 6.1 Líneas base

En este escenario las líneas base tienen una lectura peculiar y conviene
anticiparla. Si el truncamiento retuviera la dirección informativa, el
PSBPM-FD debería superar a la persistencia en la componente predecible y
empatar en las demás. Si la descartara, **todos los métodos —incluidas las
líneas base— quedarían igualados**, porque los scores retenidos serían ruido
blanco: la media incondicional sería la predicción óptima y no habría nada que
ganar.

Que las líneas base se acerquen al PSBPM-FD no es aquí un mal resultado del
modelo: es la medición del escenario.

In [ ]:
# Líneas base a h=1: el piso que el PSBP-FD debe superar.
baselines_df = tabla_baselines(SCORES_STD, T0, estandarizador=scores_standardizer,
                               fpca=fpca, X_obs=X_true, tau=grilla, h=1)
baselines_df.to_csv(PATHS["out_report"] / "30_baselines_test.csv")
display(baselines_df.style.format("{:.4f}", na_rep="—")
        .set_caption("Líneas base — bloque de prueba, h=1, contra la curva VERDADERA"))

In [ ]:
informe = verificar_contrato(PATHS)
print(f"contrato_ok = {informe['contrato_ok']}   "
      f"(M={informe['M']}, K={informe['K']}, T0={informe['T0']}, "
      f"n_components={informe['n_components']})")
print(f"estandarizador ajustado con {informe.get('estandarizador_n_ajuste')} filas (T0={informe['T0']})")
print(f"verificación FPCA: todo_ok = {informe['verificacion_fpca']['todo_ok']}")
if not informe["contrato_ok"]:
    print("\nPROBLEMAS:")
    for p in informe.get("problemas", []):
        print(f"  - {p}")

print(f"\n{'='*66}\nListo. Siguiente paso, en MATLAB desde esta carpeta:\n"
      f"  >> psbp_fd_iteracion\n"
      f"EXPERIMENT_ID = {EXPERIMENT_ID}\n"
      f"M = {M_fpca}  ·  ¿retiene la dirección informativa? "
      f"{'SÍ (caso nulo)' if RETIENE else 'NO (el escenario muerde)'}\n"
      f"{'='*66}")